# Spatiotemporal similarity with annual satellite embeddings

This notebook extends `Similarity-Embeddings.ipynb` from a single-year spatial match to two time-aware views:

1. **Fixed-state similarity through time** — where each year looks like the reference point in one reference year. This is useful for finding when/where a condition such as a recent clearcut appears.
2. **Reference-trajectory similarity** — where each pixel follows the reference point through the same years. This is useful for finding places with similar temporal evolution, not just a similar snapshot.

The similarity helper computes cosine similarity across embedding bands. If the embedding vectors are already unit-normalized, this is equivalent to the dot-product flow in the original notebook.


## 1. Initialize Earth Engine

In [ ]:
import ee

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

## 2. Imports and analysis settings

In [ ]:
from pathlib import Path

import geemap
import geopandas as gpd

EMBEDDING_COLLECTION = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"

# AlphaEarth annual embeddings are commonly used as an annual time series.
# Edit this list if a different temporal window is needed.
YEARS = list(range(2017, 2025))
REFERENCE_YEAR = 2023

SCALE_METERS = 10
EXPORT_CRS = "EPSG:5070"

MEAN_SIMILARITY_THRESHOLD = 0.85
MIN_SIMILARITY_THRESHOLD = 0.75

## 3. Load the Florida analysis extent

In [ ]:
def project_path(path: str) -> Path:
    candidate = Path(path)
    if candidate.exists():
        return candidate
    return Path("..") / candidate


extent_path = project_path("config/extent.geojson")
florida_gdf = gpd.read_file(extent_path).to_crs("EPSG:4326")
florida = ee.Geometry(florida_gdf.geometry.iloc[0].__geo_interface__)

florida_gdf[["name", "fips"]].head()

## 4. Define annual embedding helpers

In [ ]:
embeddings = ee.ImageCollection(EMBEDDING_COLLECTION)


def annual_embedding(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    image = (
        embeddings
        .filterDate(start, end)
        .filterBounds(florida)
        .mosaic()
        .clip(florida)
    )

    return image.set("year", year).set("system:time_start", start.millis())


reference_year_embedding = annual_embedding(REFERENCE_YEAR)
embedding_bands = reference_year_embedding.bandNames()

{
    "band_count": embedding_bands.size().getInfo(),
    "first_bands": embedding_bands.slice(0, 5).getInfo(),
}

## 5. Pick a reference point and comparison points

The default reference point matches the clearcut example from the original notebook. Update these coordinates to inspect other stands or disturbance types.


In [ ]:
# Example Florida point, clearcut stand in the Ocala area.
reference_point = ee.Geometry.Point([-81.8048667, 29.256675])

# Optional comparison points for the time-series table below.
comparison_points = ee.FeatureCollection([
    ee.Feature(reference_point, {"name": "reference_clearcut"}),
    ee.Feature(
        ee.Geometry.Point([-81.8039417, 29.26155]),
        {"name": "nearby_mature"},
    ),
])

## 6. Similarity functions

In [ ]:
def reference_vector(image: ee.Image, point: ee.Geometry) -> ee.Dictionary:
    sample = image.sample(
        region=point,
        scale=SCALE_METERS,
        numPixels=1,
        geometries=True,
    ).first()

    return sample.toDictionary(embedding_bands)


def constant_vector_image(vector: ee.Dictionary) -> ee.Image:
    values = vector.values(embedding_bands)
    return ee.Image.constant(values).rename(embedding_bands)


def cosine_similarity(image: ee.Image, vector: ee.Dictionary) -> ee.Image:
    vector_image = constant_vector_image(vector)
    numerator = image.multiply(vector_image).reduce(ee.Reducer.sum())
    image_norm = image.pow(2).reduce(ee.Reducer.sum()).sqrt()
    vector_norm = vector_image.pow(2).reduce(ee.Reducer.sum()).sqrt()

    return numerator.divide(image_norm.multiply(vector_norm)).rename("similarity")

## 7. Inspect the reference-year embedding vector

In [ ]:
reference_year_vector = reference_vector(reference_year_embedding, reference_point)
reference_year_vector.getInfo()

## 8. Fixed-state similarity through time

This compares every annual image to the same reference vector sampled at `reference_point` in `REFERENCE_YEAR`. Use it to ask: **where did the landscape look like this reference condition, and in which years?**


In [ ]:
def fixed_state_similarity_image(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, 1, 1)

    return (
        cosine_similarity(annual_embedding(year), reference_year_vector)
        .set("year", year)
        .set("system:time_start", start.millis())
    )


fixed_state_similarity_images = [fixed_state_similarity_image(year) for year in YEARS]
fixed_state_similarity = ee.ImageCollection.fromImages(fixed_state_similarity_images)
fixed_state_similarity_stack = ee.Image.cat(*[
    image.rename(f"similarity_{year}")
    for image, year in zip(fixed_state_similarity_images, YEARS)
])

fixed_state_similarity_stack.bandNames().getInfo()

### Fixed-state point time series

In [ ]:
fixed_state_point_samples = fixed_state_similarity_stack.sampleRegions(
    collection=comparison_points,
    scale=SCALE_METERS,
    geometries=True,
)

fixed_state_point_df = geemap.ee_to_df(fixed_state_point_samples)
fixed_state_columns = [f"similarity_{year}" for year in YEARS]

fixed_state_point_df.set_index("name")[fixed_state_columns].T.rename_axis("year")

## 9. Reference-trajectory similarity

This compares each year to the reference point from that **same year**, then summarizes across years. The mean layer finds places with consistently similar annual states; the minimum layer is stricter and highlights persistent matches.


In [ ]:
def trajectory_similarity_image(year: int) -> ee.Image:
    image = annual_embedding(year)
    year_vector = reference_vector(image, reference_point)
    start = ee.Date.fromYMD(year, 1, 1)

    return (
        cosine_similarity(image, year_vector)
        .set("year", year)
        .set("system:time_start", start.millis())
    )


trajectory_similarity_images = [trajectory_similarity_image(year) for year in YEARS]
trajectory_similarity = ee.ImageCollection.fromImages(trajectory_similarity_images)
trajectory_similarity_stack = ee.Image.cat(*[
    image.rename(f"same_year_similarity_{year}")
    for image, year in zip(trajectory_similarity_images, YEARS)
])

trajectory_mean = trajectory_similarity.mean().rename("trajectory_similarity_mean")
trajectory_minimum = trajectory_similarity.min().rename("trajectory_similarity_minimum")
trajectory_std_dev = (
    trajectory_similarity
    .reduce(ee.Reducer.stdDev())
    .rename("trajectory_similarity_std_dev")
)

persistent_mask = trajectory_mean.gt(MEAN_SIMILARITY_THRESHOLD).multiply(
    trajectory_minimum.gt(MIN_SIMILARITY_THRESHOLD)
)
persistent_trajectory_similarity = trajectory_mean.updateMask(persistent_mask)

ee.Image.cat(trajectory_mean, trajectory_minimum, trajectory_std_dev).bandNames().getInfo()

### Reference-trajectory point time series

In [ ]:
trajectory_point_samples = trajectory_similarity_stack.sampleRegions(
    collection=comparison_points,
    scale=SCALE_METERS,
    geometries=True,
)

trajectory_point_df = geemap.ee_to_df(trajectory_point_samples)
trajectory_columns = [f"same_year_similarity_{year}" for year in YEARS]

trajectory_point_df.set_index("name")[trajectory_columns].T.rename_axis("year")

## 10. Map the spatial and spatiotemporal products

In [ ]:
similarity_vis = {
    "min": 0.6,
    "max": 1.0,
    "palette": ["black", "blue", "cyan", "yellow", "red"],
}
stability_vis = {
    "min": 0.0,
    "max": 0.15,
    "palette": ["white", "orange", "red"],
}

m = geemap.Map()
m.centerObject(reference_point, 10)
m.add_basemap("SATELLITE")
m.addLayer(
    fixed_state_similarity_stack.select(f"similarity_{REFERENCE_YEAR}"),
    similarity_vis,
    f"Fixed-state similarity ({REFERENCE_YEAR})",
)
m.addLayer(
    trajectory_mean,
    similarity_vis,
    f"Trajectory mean similarity ({YEARS[0]}-{YEARS[-1]})",
)
m.addLayer(
    trajectory_minimum,
    similarity_vis,
    f"Trajectory minimum similarity ({YEARS[0]}-{YEARS[-1]})",
)
m.addLayer(trajectory_std_dev, stability_vis, "Trajectory similarity std. dev.")
m.addLayer(persistent_trajectory_similarity, similarity_vis, "Persistent trajectory match")
m.addLayer(comparison_points, {"color": "white"}, "Reference/comparison points")

m

## 11. Optional export

In [ ]:
# Uncomment to export both the annual stacks and the trajectory summaries to Drive.
# export_image = ee.Image.cat(
#     fixed_state_similarity_stack,
#     trajectory_similarity_stack,
#     trajectory_mean,
#     trajectory_minimum,
#     trajectory_std_dev,
#     persistent_trajectory_similarity.rename("persistent_trajectory_similarity"),
# )
#
# task = ee.batch.Export.image.toDrive(
#     image=export_image,
#     description=f"alphaearth_spatiotemporal_similarity_{YEARS[0]}_{YEARS[-1]}",
#     folder="artemis_exports",
#     fileNamePrefix=f"alphaearth_spatiotemporal_similarity_{YEARS[0]}_{YEARS[-1]}",
#     region=florida,
#     scale=SCALE_METERS,
#     crs=EXPORT_CRS,
#     maxPixels=1e13,
# )
#
# task.start()
# task.status()

## Interpretation notes

- Use **fixed-state similarity** when the question is about a condition from one year appearing elsewhere or in other years.
- Use **trajectory mean/minimum similarity** when the question is about places that behave like the reference point over the whole period.
- Increase `MIN_SIMILARITY_THRESHOLD` to require stronger temporal persistence; lower it to allow short-lived deviations.
- The standard-deviation layer helps separate stable analogs from places that match only intermittently.
